# Chapter 20: Graphs

The programs from Chapter 20 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Three Ways to Store a Graph

`representations.py`

In [1]:
"""One small undirected graph, written down three ways."""

edges = [("A", "B"), ("A", "C"), ("B", "C"), ("C", "D")]
print("edge list:", edges)

adjacency = {}
for u, v in edges:
    adjacency.setdefault(u, []).append(v)
    adjacency.setdefault(v, []).append(u)
print("adjacency list:")
for v, nbrs in sorted(adjacency.items()):
    print(f"  {v}: {nbrs}")

names = sorted(adjacency)
position = {v: i for i, v in enumerate(names)}
matrix = [[0] * len(names) for _ in names]
for u, v in edges:
    matrix[position[u]][position[v]] = matrix[position[v]][position[u]] = 1
print("adjacency matrix:")
print("    " + " ".join(names))
for name, row in zip(names, matrix):
    print(f"  {name} " + " ".join(map(str, row)))

edge list: [('A', 'B'), ('A', 'C'), ('B', 'C'), ('C', 'D')]
adjacency list:
  A: ['B', 'C']
  B: ['A', 'C']
  C: ['A', 'B', 'D']
  D: ['C']
adjacency matrix:
    A B C D
  A 0 1 1 0
  B 1 0 1 0
  C 1 1 0 1
  D 0 0 1 0


`memory_sizes.py`

In [2]:
"""How many entries each representation stores for a sparse graph:
V vertices, each joined to about 4 others."""

print(
    f"{'vertices':>9} {'edges':>8} {'adjacency list':>15} "
    f"{'matrix cells':>16}"
)
for v in [100, 1_000, 10_000, 100_000]:
    e = 2 * v  # about 4 neighbors each: every edge has two ends
    print(f"{v:>9,} {e:>8,} {2 * e:>15,} {v * v:>16,}")

 vertices    edges  adjacency list     matrix cells
      100      200             400           10,000
    1,000    2,000           4,000        1,000,000
   10,000   20,000          40,000      100,000,000
  100,000  200,000         400,000   10,000,000,000


## A Graph Class

`graph.py`

In [3]:
"""A graph as an adjacency map: each vertex maps to its neighbors, and
each neighbor to the weight of the edge (1 when weights don't matter)."""


class Graph:
    def __init__(self, directed=False):
        self.directed = directed
        self._adj = {}  # vertex -> {neighbor: weight}

    @classmethod
    def from_edges(cls, edges, directed=False):
        """Edges as (u, v) or (u, v, weight)."""
        g = cls(directed)
        for edge in edges:
            g.add_edge(*edge)
        return g

    def add_vertex(self, v):
        self._adj.setdefault(v, {})

    def add_edge(self, u, v, weight=1):
        self.add_vertex(u)
        self.add_vertex(v)
        self._adj[u][v] = weight
        if not self.directed:
            self._adj[v][u] = weight  # an undirected edge goes both ways

    def has_edge(self, u, v):
        return v in self._adj.get(u, {})

    def neighbors(self, v):
        return self._adj[v].keys()

    def weight(self, u, v):
        return self._adj[u][v]

    def vertices(self):
        return list(self._adj)

    def edges(self):
        """Every edge once, as (u, v, weight)."""
        seen = set()
        for u, nbrs in self._adj.items():
            for v, w in nbrs.items():
                if self.directed or (v, u) not in seen:
                    seen.add((u, v))
                    yield u, v, w

    def degree(self, v):
        return len(self._adj[v])

    def __len__(self):
        return len(self._adj)

## Checking an Edge

`bench_edge_query.py`

In [4]:
"""Is there an edge from u to v? With neighbor lists, neighbor sets
and a matrix, as the graph grows (each vertex has 10 neighbors)."""

import random

import bench

rng = random.Random(2026)


def make(n):
    nbrs = [rng.sample(range(n), 10) for _ in range(n)]
    lists = nbrs
    sets = [set(x) for x in nbrs]
    matrix = [[False] * n for _ in range(n)]
    for u, row in enumerate(nbrs):
        for v in row:
            matrix[u][v] = True
    return lists, sets, matrix


sizes = [250, 500, 1_000, 2_000]
built = {n: make(n) for n in sizes}
results = {}
for i, name in enumerate(["lists", "sets", "matrix"]):
    if name == "matrix":

        def query(r):
            return r[7][-1]  # row 7, the last column
    else:

        def query(r):
            return -1 in r[7]  # not there: a list scans all 10

    results[name] = bench.doubling(
        query, lambda n, i=i: (built[n][i],), sizes
    )
bench.table(results)

         n           lists            sets          matrix
       250          128 ns         70.3 ns         70.5 ns
       500     133 ns x1.0    68.5 ns x1.0    70.9 ns x1.0
     1,000     131 ns x1.0    68.1 ns x1.0    72.5 ns x1.0
     2,000     133 ns x1.0    70.3 ns x1.0    72.2 ns x1.0
